# Очистка данных и Feature Engineering

Идея и что я хочу тут реализовать

1. Загрузить и оптимизировать по памяти 12.8 млн строк дампа IMDb.
2. Очистить данные от невалидных значений.
3. Сгенерировать антифрод-признаки (`log_votes`, `rating_to_log_votes`).
4. Сформировать таргет редкого класса (0.17%) и сохранить очищенный датасет.

In [2]:
#Основные импорты
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Проверяем

print(f"Pandas: {pd.__version__}, NumPy: {np.__version__}")

Pandas: 3.0.6, NumPy: 2.5.4


### 1. Загрузка метаданных (title.basics) и оптимизация памяти
В сыром дампе 12.8 млн записей. Оптимизируем потребление RAM:
- Читаем только необходимые столбцы (`usecols`)
- Указываем строгие типы данных (`dtype`), переводим `titleType` в `category`, если не указывать просто все будут обджект, а это самый плохой тип
- Сразу фильтруем только полнометражные фильмы (`titleType == 'movie'`) и освобождаем память.

In [3]:
basics = pd.read_csv(
    'data/title.basics.tsv.gz',
    sep='\t',
    na_values=r'\N',
    usecols=['tconst', 'titleType', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres'],
    dtype={
        'tconst': 'str',
        'titleType': 'category',
        'primaryTitle': 'str',
        'startYear': 'str',
        'runtimeMinutes': 'str',
        'genres': 'str'
    }
)

# Только фильмы
movies = basics[basics['titleType'] == 'movie'].copy()

# Память
del basics

print(f"Полнометражных фильмов отобрано: {len(movies):,}")
movies.head()

Полнометражных фильмов отобрано: 759,363


,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres
8,tt0000009,movie,Miss Jerry,1894,45,Romance
144,tt0000147,movie,The Corbett-Fitzsimmons Fight,1897,100,"Documentary,News,Sport"
498,tt0000502,movie,Bohemios,1905,100,NaN
570,tt0000574,movie,The Story of the Kelly Gang,1906,70,"Action,Adventure,Biography"
587,tt0000591,movie,The Prodigal Son,1907,90,Drama
